# 01 — Data Collection

Fetches and merges all data sources needed for the Prop 13 tax disparity analysis:
- **CA BOE 2022-23**: Total secured assessed values by county (embedded)
- **Zillow ZHVI**: County-level median home values (downloaded)
- **Census ACS 5-Year 2022**: Owner-occupied units and median household income (fetched)

Output: `data/processed/county_prop13.csv`

In [1]:
import sys
from pathlib import Path

# Add src to path when running from notebooks/ directory
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from ca_data import (
    get_boe_data,
    fetch_zillow_county_data,
    fetch_census_data,
    compute_tax_metrics,
)

OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
# 1. BOE assessed values
boe = get_boe_data()
print(f'BOE data: {len(boe)} counties')
boe.head()

BOE data: 58 counties


,county,fips,county_fips,boe_total_av_millions,residential_fraction,boe_residential_av_millions
0,Alameda,06001,001,352000,0.72,253440.0
1,Alpine,06003,003,500,0.85,425.0
2,Amador,06005,005,10000,0.85,8500.0
3,Butte,06007,007,25000,0.78,19500.0
4,Calaveras,06009,009,12000,0.85,10200.0


In [3]:
# 2. Zillow market values
zillow = fetch_zillow_county_data()
print(f'Zillow data: {len(zillow)} CA counties')
zillow.head()

  Using Zillow data through 2026-02-28
Zillow data: 58 CA counties


,fips,zillow_median_value
0,06037,8.788510e+05
4,06073,9.284592e+05
5,06059,1.169743e+06
9,06065,6.032127e+05
13,06071,5.471532e+05


In [4]:
# 3. Census ACS data
census = fetch_census_data()
print(f'Census data: {len(census)} CA counties')
census.head()

Fetching Census ACS 5-Year 2022 data ...


Census data: 58 CA counties


,fips,owner_occupied_units,median_income,census_median_value,census_aggregate_value,census_aggregate_taxes
0,06001,317451,122488,999200,366023813600,2754706900
1,06003,360,101125,463900,235732900,1487300
2,06005,12440,74853,409600,6285757500,38356100
3,06007,48424,66085,371600,20545982200,146618200
4,06009,14114,77526,404200,6560376100,48768300


In [5]:
# 4. Merge all sources
df = boe.merge(zillow, on='fips', how='left')
df = df.merge(census, on='fips', how='left')

# Prefer Zillow for market value; fall back to Census
df['zillow_median_value'] = df['zillow_median_value'].fillna(df['census_median_value'])

print(f'Merged: {len(df)} counties, {df.columns.tolist()}')

Merged: 58 counties, ['county', 'fips', 'county_fips', 'boe_total_av_millions', 'residential_fraction', 'boe_residential_av_millions', 'zillow_median_value', 'owner_occupied_units', 'median_income', 'census_median_value', 'census_aggregate_value', 'census_aggregate_taxes']


In [6]:
# 5. Compute Prop 13 tax metrics
df = compute_tax_metrics(df)

# Save
out_path = OUT / 'county_prop13.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')
print(f'Statewide annual tax gap estimate: ${df["tax_gap_annual_millions"].sum():,.0f}M')

Saved to /Users/mckornfield/repo/california-stats/data/processed/county_prop13.csv
Statewide annual tax gap estimate: $25,629M


In [7]:
# Preview key columns
preview = df[['county', 'residential_av_millions', 'market_value_millions',
              'assessment_ratio', 'tax_gap_annual_millions', 'tax_gap_per_household']]
preview = preview.sort_values('assessment_ratio')
preview.style.format({
    'residential_av_millions': '${:,.0f}M',
    'market_value_millions': '${:,.0f}M',
    'assessment_ratio': '{:.2%}',
    'tax_gap_annual_millions': '${:,.0f}M',
    'tax_gap_per_household': '${:,.0f}',
})

,county,residential_av_millions,market_value_millions,assessment_ratio,tax_gap_annual_millions,tax_gap_per_household
52,Trinity,$684M,"$2,328M",29.39%,$18M,"$4,656"
40,San Mateo,"$154,305M","$292,050M",52.84%,"$1,515M","$9,628"
41,Santa Barbara,"$45,580M","$85,327M",53.42%,$437M,"$5,593"
54,Tuolumne,"$4,579M","$8,361M",54.77%,$42M,"$2,448"
2,Amador,"$3,487M","$6,286M",55.47%,$31M,"$2,475"
22,Mendocino,"$6,812M","$12,184M",55.91%,$59M,"$2,808"
27,Napa,"$18,578M","$33,117M",56.10%,$160M,"$4,986"
5,Colusa,"$1,129M","$2,009M",56.21%,$10M,"$2,065"
21,Mariposa,"$1,317M","$2,340M",56.28%,$11M,"$2,008"
43,Santa Cruz,"$35,720M","$62,778M",56.90%,$298M,"$5,145"
